# Model Experiments — Evaluation & Comparison

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

ROOT = Path('..')
print('Imports OK')

## 1. Load Evaluation Reports

In [ ]:
def load_report(domain):
    path = ROOT / 'models' / domain / 'evaluation_report.json'
    if not path.exists():
        print(f'No report for {domain} — run training first')
        return None
    with open(path) as f:
        return json.load(f)

cr_report = load_report('credit_risk')
ni_report = load_report('network_intrusion')

if cr_report and ni_report:
    print('Both reports loaded successfully')
    print('\nCredit Risk metrics:', cr_report['scalar_metrics'])
    print('\nNetwork Intrusion metrics:', ni_report['scalar_metrics'])

## 2. Metrics Comparison Table

In [ ]:
if cr_report and ni_report:
    metrics_df = pd.DataFrame({
        'Credit Risk': cr_report['scalar_metrics'],
        'Network Intrusion': ni_report['scalar_metrics']
    }).T[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
    metrics_df.round(4)

## 3. Confusion Matrices

The confusion matrix shows the four prediction outcomes:
- **True Negative (TN)**: Correctly predicted safe
- **False Positive (FP)**: Wrongly flagged as risky (false alarm)
- **False Negative (FN)**: Missed a real risk (dangerous in finance!)
- **True Positive (TP)**: Correctly predicted risky

In [ ]:
if cr_report and ni_report:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for ax, report, title in zip(
        axes,
        [cr_report, ni_report],
        ['Credit Risk', 'Network Intrusion']
    ):
        cm = np.array(report['confusion_matrix'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Pred Neg', 'Pred Pos'],
                    yticklabels=['Actual Neg', 'Actual Pos'],
                    cbar=False)
        tn, fp, fn, tp = cm.ravel()
        ax.set_title(f'{title}\nROC-AUC={report["scalar_metrics"]["roc_auc"]:.4f}')

    plt.tight_layout()
    plt.savefig('../models/confusion_matrices.png', bbox_inches='tight')
    plt.show()

## 4. SHAP Explainability Plots

In [ ]:
from IPython.display import Image, display

for domain in ['credit_risk', 'network_intrusion']:
    shap_path = ROOT / 'models' / domain / 'shap_summary.png'
    if shap_path.exists():
        print(f'\n=== {domain.replace("_", " ").title()} SHAP Summary ===')
        display(Image(str(shap_path), width=800))
    else:
        print(f'SHAP plots not found for {domain} — run training first')